# MongoDB Atlas VFS for LangChain Deep Agents

**Building a Multi-Agent Pipeline Where Nothing Gets Lost**

This notebook walks through a multi-agent research pipeline that reads a real federal rulemaking docket (DOE air-cleaner efficiency standards), finds cross-document discrepancies no single file contains, and survives being killed mid-run.

## What you'll see

1. **Beat 1 — Discovery across formats**: hybrid search finds concepts across PDFs, spreadsheets, and DOCX files where literal grep cannot
2. **Beat 2 — Multi-agent pipeline**: four sub-agents coordinate through a shared durable workspace (via subprocess)
3. **Beat 3 — The kill test**: kill the pipeline mid-run, resume it, same result (via subprocess)

## Prerequisites

- MongoDB Atlas cluster (M0+ for dev, M10+ for Search/Vector Search)
- AWS account with S3 bucket
- OpenAI API key
- Corpus seeded via `python scripts/00_seed_corpus.py`

In [ ]:
# Install dependencies
%pip install -q langchain-mongodb-deepagents-vfs deepagents langchain-openai pymongo boto3 python-dotenv

In [ ]:
import os
from datetime import datetime

from dotenv import load_dotenv

load_dotenv("../.env")

# Verify required environment variables
required = ["MONGODB_URI", "S3_BUCKET_NAME", "OPENAI_API_KEY"]
missing = [k for k in required if not os.environ.get(k)]
if missing:
    raise OSError(f"Missing environment variables: {missing}")

# Generate unique run IDs so the notebook is re-runnable
_ts = datetime.now().strftime("%m%d-%H%M%S")
PIPELINE_RUN_ID = f"notebook-{_ts}"
KILL_RUN_ID = f"notebook-kill-{_ts}"

print("Environment configured:")
print(f"  S3 bucket: {os.environ['S3_BUCKET_NAME']}")
print(f"  AWS region: {os.environ.get('AWS_REGION', 'us-east-1')}")
print(f"  MongoDB: {'***' + os.environ['MONGODB_URI'][-20:]}")
print(f"  Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"  Kill test run ID: {KILL_RUN_ID}")

## Beat 1 — Discovery Across Formats

Three arms compare how different search approaches handle a mixed-format corpus:

| Arm | Method | Limitation |
|-----|--------|------------|
| **ripgrep** | Literal string match over local files | Cannot read PDF, XLSX, DOCX |
| **StoreBackend** | `StoreBackend.grep` — literal substring in Python | Never calls `MongoDBStore.search()`, so vector search is unused |
| **MongoFilesystemBackend** | Hybrid `$rankFusion` (50/50 fulltext + vector) | Chunk-level, cross-vocabulary |

**Query:** *"How is air cleaner efficiency measured, and do the state standards use the same metric as DOE?"*

The thesis here is **parsing and vocabulary**, not "vector beats grep." BM25 is underrated for agentic search — the two claims that survive the critique are:
1. BM25 cannot read the spreadsheets or DOCX uploads at all (extraction, not ranking)
2. Four parties using four names for one metric is not a tuning problem

In [ ]:
import time
from pathlib import Path

from langchain_mongodb_deepagents_vfs import MongoFilesystemBackend

APP_NAME = "devrel-tutorial-deepagents-langchain-vfs"

QUERY = (
    "How is air cleaner efficiency measured, and do the state standards "
    "use the same metric as DOE?"
)

# Append appName for DevRel tracking
mongodb_uri = os.environ["MONGODB_URI"]
if "appName=" not in mongodb_uri and "appname=" not in mongodb_uri:
    sep = "&" if "?" in mongodb_uri else "?"
    mongodb_uri = f"{mongodb_uri}{sep}appName={APP_NAME}"

# MongoFilesystemBackend — hybrid $rankFusion search
with MongoFilesystemBackend(
    s3_bucket_name=os.environ["S3_BUCKET_NAME"],
    mongodb_connection_string=mongodb_uri,
    s3_prefix="corpus/",
    aws_region=os.environ.get("AWS_REGION", "us-east-1"),
    debug=True,
) as backend:
    # Warmup grep — blocks until initial sync completes
    backend.grep("warmup")

    # Health check — three documented failure modes are SILENT
    assert not backend.init_errors, f"Init errors: {backend.init_errors}"
    report = backend.initial_sync_report
    assert (
        report and report.failed == 0
    ), f"Sync failures: {report.failed if report else 'no report'}"
    print(
        f"Sync: {report.seen} seen, {report.processed} processed, "
        f"{report.skipped} skipped, {report.failed} failed"
    )

    # Run 3x for reproducibility
    for run_num in range(1, 4):
        t0 = time.monotonic()
        result = backend.grep(QUERY)
        elapsed = time.monotonic() - t0

        matches = result.matches or []
        hit_types = {Path(m["path"]).suffix.lower() for m in matches}

        # Check vocabulary coverage
        vocab = {"IEF": False, "smoke CADR": False, "PM2.5": False, "CADR/W": False}
        for m in matches:
            for term in vocab:
                if term.lower() in m["text"].lower():
                    vocab[term] = True

        print(f"\nRun {run_num}: {len(matches)} matches in {elapsed:.2f}s")
        print(f"  File types: {sorted(hit_types)}")
        print(f"  Vocabulary: {vocab}")
        for m in matches[:3]:
            print(f"  {m['path']}:{m['line']} — {m['text'][:80]}...")

## Beat 2 — Multi-Agent Pipeline

The pipeline runs four sub-agents that share a durable workspace:

```
coordinator
├── writes  workspace/<run_id>/manifest.json     ← run receipt
├── task → proposal-reader  → findings/proposal.md
├── task → adoption-reader  → findings/adopted.md
├── task → numbers-reader   → findings/numbers.md
└── task → writer           → reads three by path → memo.md
```

**Two planes, two guarantees:**
- **Corpus** (discovery): searched by meaning via `grep`, eventually consistent
- **Workspace** (coordination): read/written by exact path, read-after-write consistent

The writer reads by known path — immediate, deterministic, no watcher in the coordination path.

> Beats 2 and 3 shell out because a background watcher thread, sub-agent fan-out, and a process kill are not notebook-shaped.

In [ ]:
import subprocess

# Run the full pipeline (cold run)
result = subprocess.run(
    ["python", "../scripts/03_pipeline.py", "--run-id", PIPELINE_RUN_ID],
    capture_output=True,
    text=True,
    timeout=300,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")

## Beat 3 — The Kill Test

An intern that can't be interrupted isn't an intern.

We kill the pipeline after 2 of 4 stages, then resume with the same `run_id`. The coordinator reads `manifest.json`, sees two stages complete, skips them, and runs the rest.

**What to watch for:**
- The resume skips completed stages
- The final memo is the same as a cold run
- `resumed + killed ≈ cold` in tokens and cost

In [ ]:
# Step 1: Start a run and kill it after 2 stages
result = subprocess.run(
    [
        "python",
        "../scripts/03_pipeline.py",
        "--run-id",
        KILL_RUN_ID,
        "--kill-after",
        "2",
    ],
    capture_output=True,
    text=True,
    timeout=300,
)
print("--- Kill run output ---")
print(result.stdout)
# Expect non-zero exit (SIGKILL)
print(f"Exit code: {result.returncode} (expected: non-zero from SIGKILL)")

In [ ]:
# Step 2: Resume the killed run
result = subprocess.run(
    ["python", "../scripts/04_resume.py", "--run-id", KILL_RUN_ID],
    capture_output=True,
    text=True,
    timeout=300,
)
print("--- Resume output ---")
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:] if result.stderr else "")

## What's next

**Before production, you'd want:**

1. **Silent-degradation monitoring** — non-Atlas MongoDB silently falls back to regex; an unavailable embedding API silently falls back to full-text only; a partial sync looks healthy while being incomplete. All three produce no error.

2. **The 64 MiB read cap** — oversized objects are skipped, counted in `SyncReport.failed` during initial sync but only *logged* by the watchers.

3. **Write→grep lag** — budget for the watcher interval (10s) plus Atlas indexing time. Don't rely on a file being greppable immediately after writing it.

4. **Access control is the application's job** — `s3_prefix` isolation and `FilesystemPermission` are the tools available; the backend does not enforce ACLs.

5. **Cost tracking per task** — the `manifest.json` run receipt records tokens and USD per stage, making cost reconciliation possible at the trace level rather than coarse spend totals.